# Import Libraries

In [1]:
import pandas as pd
import numpy as np
import spacy
import os
from tqdm import tqdm 
from sklearn.metrics import classification_report
import os
import requests
from openai import OpenAI, RateLimitError
import time
import random

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full content in each cell
pd.set_option('display.width', 1000)  # Set max width

# Load spaCy's English model
nlp = spacy.load('en_core_web_sm')

# Pre-Processing

In [2]:
label_mapper = {
    'knowledge' : 0,
    'comprehension' : 1,
    'application' : 2,
    'analysis' : 3,
    'synthesis' : 4,
    'evaluation' : 5
}

mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}

q_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset4.csv')
queries = q_df['question']
q_df['label'] = q_df['label'].str.lower()
q_df['label'] = q_df['label'].replace(mapping)
label = q_df['label'].str.lower().map(label_mapper)
print(q_df['label'].value_counts())

label
synthesis        29
knowledge        22
evaluation       21
comprehension    20
analysis         19
application      15
Name: count, dtype: int64


# API Setup

In [3]:
# Sonar
 
api_key = os.environ.get("PERPLEXITY_API_KEY")

if api_key:
    print('successful')

url = "https://api.perplexity.ai/chat/completions"
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

successful


In [24]:
# Groq

api_key = os.environ.get("GROQ_API_KEY")

if api_key:
    print('successful')

groq_client = OpenAI(
    base_url = "https://api.groq.com/openai/v1",
    api_key = api_key
)

successful


# Zero-Shot

## GPT-OSS-120B

### Assign Labels

In [ ]:
zso_pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Given the query below, classify which label it belongs.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]
                
                query : {query}""",
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="openai/gpt-oss-20b",
        )

        reply = chat_completion.choices[0].message.content.lower()

    zso_pred_labels.append(reply.lower())

100%|██████████| 126/126 [08:06<00:00,  3.86s/it]


In [18]:
print(classification_report(label , [label_mapper[key.lower()] for key in zso_pred_labels]))

              precision    recall  f1-score   support

           0       0.88      1.00      0.94        22
           1       0.82      0.70      0.76        20
           2       0.47      0.47      0.47        15
           3       0.79      0.58      0.67        19
           4       0.69      0.86      0.77        29
           5       0.89      0.81      0.85        21

    accuracy                           0.76       126
   macro avg       0.76      0.74      0.74       126
weighted avg       0.77      0.76      0.76       126



## LLAMA4-Scout

In [19]:
zsl_pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Given the query below, classify which label it belongs.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]
                
                query : {query}""",
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()

    zsl_pred_labels.append(reply.lower())

100%|██████████| 126/126 [09:09<00:00,  4.36s/it]


In [20]:
print(classification_report(label , [label_mapper[key.lower()] for key in zsl_pred_labels]))

              precision    recall  f1-score   support

           0       0.88      0.68      0.77        22
           1       0.61      0.70      0.65        20
           2       0.42      0.53      0.47        15
           3       0.71      0.63      0.67        19
           4       0.78      0.86      0.82        29
           5       0.94      0.81      0.87        21

    accuracy                           0.72       126
   macro avg       0.72      0.70      0.71       126
weighted avg       0.74      0.72      0.73       126



## INSTRUCTION PROMPT

## GPT-OSS-120B

In [ ]:
ipo_pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Given the query below, classify which label it belongs.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Definations:
                    
                    1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
                    2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
                    3. Application: Using learned information in new concrete situations to solve problems
                    4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
                    5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
                    6. Evaluation: Making judgments based on criteria and standards through checking and critiquing

                    Now classify the following question:

                    Question: {query}
                    """,
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="openai/gpt-oss-20b",
        )

        reply = chat_completion.choices[0].message.content.lower()

    ipo_pred_labels.append(reply.lower())

  0%|          | 0/126 [00:00<?, ?it/s]


BadRequestError: Error code: 400 - {'error': {'message': 'Organization has been restricted. Please reach out to support if you believe this was in error.', 'type': 'invalid_request_error', 'code': 'organization_restricted'}}

In [ ]:
print(classification_report(label , [label_mapper[key.lower()] for key in ipo_pred_labels]))

              precision    recall  f1-score   support

           0       0.85      1.00      0.92        22
           1       0.67      0.70      0.68        20
           2       0.36      0.33      0.34        15
           3       0.85      0.58      0.69        19
           4       0.68      0.86      0.76        29
           5       1.00      0.71      0.83        21

    accuracy                           0.73       126
   macro avg       0.73      0.70      0.70       126
weighted avg       0.75      0.73      0.73       126



## LLAMA4-Scout

In [22]:
ipl_pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Given the query below, classify which label it belongs.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Definations:
                    
                    1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
                    2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
                    3. Application: Using learned information in new concrete situations to solve problems
                    4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
                    5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
                    6. Evaluation: Making judgments based on criteria and standards through checking and critiquing

                    Now classify the following question:

                    Question: {query}
                    """,
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()

    ipl_pred_labels.append(reply.lower())

100%|██████████| 126/126 [09:09<00:00,  4.36s/it]


In [23]:
print(classification_report(label , [label_mapper[key.lower()] for key in ipl_pred_labels]))

              precision    recall  f1-score   support

           0       0.91      0.91      0.91        22
           1       0.75      0.75      0.75        20
           2       0.42      0.33      0.37        15
           3       0.81      0.68      0.74        19
           4       0.67      0.90      0.76        29
           5       0.82      0.67      0.74        21

    accuracy                           0.74       126
   macro avg       0.73      0.71      0.71       126
weighted avg       0.74      0.74      0.73       126



# Chain-of-Thought without Context

## GPT-OSS-120B

In [ ]:
coto_pred_labels = []

for query in tqdm(queries):
    # Reason

    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""
                    Given the query below, reason about which label it belongs.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Query: {query}""",
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    # Summarize and Classify
    
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""
                    Summarize the reasoning into exactly one label from.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Reasoning: {reply}

                    Answer in one word ONLY""",
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from:
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="openai/gpt-oss-20b",
        )

        reply = chat_completion.choices[0].message.content.lower()
    time.sleep(10)

    coto_pred_labels.append(reply.lower())

In [ ]:
print(classification_report(label , [label_mapper[key.lower()] for key in coto_pred_labels]))

## LLAMA4-Scout

In [ ]:
cotl_pred_labels = []

for query in tqdm(queries):
    # Reason

    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""
                    Given the query below, reason about which label it belongs.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Query: {query}""",
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    # Summarize and Classify
    
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""
                    Summarize the reasoning into exactly one label from.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Reasoning: {reply}

                    Answer in one word ONLY""",
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from:
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()
    time.sleep(10)

    cotl_pred_labels.append(reply.lower())

In [ ]:
print(classification_report(label , [label_mapper[key.lower()] for key in cotl_pred_labels]))